# Lab02 — Fine-tuning DenseNet121 trên CIFAR10

Notebook thực nghiệm Kaggle dùng cùng backend với local. So sánh learning rate chung và learning rate riêng cho backbone/head; train head mới từ ImageNet pretrained, không dùng checkpoint kết quả cũ. Các kết luận chỉ được đưa ra sau khi chạy các cell và đọc artifact.

Bật GPU và Internet, gắn `lab02_optuna_source.zip` mới dưới dạng Input. Không cần checkpoint cũ. Phần training gọi backend tổng hợp `src.optuna_dense.run`; các cell trước xác minh phương pháp, các cell sau phân tích artifact. Đây không phải các giai đoạn training độc lập theo từng cell.

Toàn bài giữ ResNet18 với recipe cũ; chỉ DenseNet đổi sang Optuna. Phần ResNet bổ sung ở cuối không chạy lại DenseNet.


## Pipeline Execution Roadmap

| Bước | Giai đoạn | Nội dung | Module | Output / Artifact |
|---|---|---|---|---|
| 1 | Setup | GPU, source, dependencies, seed | bootstrap; `experiment.seed_all` | snapshot môi trường |
| 2 | Dữ liệu | CIFAR10 official và kiểm tra checksum | `eda.read_raw` | 50.000 train pool, 10.000 test |
| 3 | EDA | Split, integrity, phân bố, pixel/spatial, feature mô tả | `eda.build`, `locked_split` | tables/figures tại outputs/eda |
| 4 | Preprocess | Input RGB 224×224 theo pretrained weights | `data.image_transform` | tensor contract, preprocessing preview thật |
| 5 | Model | Backbone frozen; fine-tune block4 + norm5 + head | `models.configure_trainable` | architecture, trainable counts |
| 6 | Thiết kế search | 5 shared + 5 separate; AdamW/cosine | `optuna_dense.suggest` | search config |
| 7 | Training | Head mới 10 epoch rồi 10 trial tối đa 20 epoch | `optuna_dense.run`, `experiment.fit` | TensorBoard, histories, checkpoints, study SQLite |
| 8 | Lựa chọn | Validation loss; min_delta/patience thống nhất | `experiment.fit(selection_metric='loss')` | best_parameters và best.pt |
| 9 | So sánh | Loss/accuracy, metrics, ngân sách thực tế | `optuna_dense.plot_results` | comparison.csv, learning_curves.png |
| 10 | Test & Load | Confusion, ROC/PR OvR, dự đoán; load lại full test | `reporting.export_evaluation`, `experiment.predict` | evaluation/, loaded/, roundtrip.json |
| 11 | Bàn giao | Đóng gói run và EDA, tải notebook có output | notebook + manifest | ZIP artifacts, registry |

Test không dùng để chọn trial; official test đã được xem trong các thí nghiệm lịch sử nên cần nêu giới hạn khi viết báo cáo.

## 1. Thiết lập Kaggle và source

Các file trong `/kaggle/input` chỉ đọc. Source được chép vào `/kaggle/working/lab02_optuna`; mọi dữ liệu tải mới, study, checkpoint và log đều ghi dưới working. Mỗi lần gọi training tạo run mới; để xem lại kết quả, chạy các cell phân tích thay vì chạy lại training.

In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, shutil, json, os
import torch
assert torch.cuda.is_available(), 'Bật GPU trong cài đặt notebook trước khi chạy.'
print(torch.__version__, torch.cuda.get_device_name(0))
WORK = Path('/kaggle/working/lab02_optuna')
# Có nhiều bộ source thì chọn đường dẫn ở đây; mặc định tự tìm.
SOURCE_ROOT = None
SOURCE_ZIP = None
required = ('src/optuna_dense.py', 'config/optuna_dense.json', 'tests/test_optuna_dense.py')
if (WORK/'outputs/runs').exists():
    assert all((WORK/p).is_file() for p in required), 'Run đã tồn tại nhưng source thiếu. Không ghi đè source của run.'
    print('Giữ nguyên source và run đã có. Không chạy lại cell TRAINING nếu chỉ xem kết quả.')
else:
    if SOURCE_ROOT is None and SOURCE_ZIP is None:
        roots = sorted({p.parent.parent for p in Path('/kaggle/input').rglob('optuna_dense.py')
                        if p.parent.name == 'src' and all((p.parent.parent/f).is_file() for f in required)})
        archives = sorted(Path('/kaggle/input').rglob('lab02_optuna_source.zip'))
        if len(roots) == 1:
            SOURCE_ROOT = roots[0]
        elif not roots and len(archives) == 1:
            SOURCE_ZIP = archives[0]
        else:
            print('Source folders:', roots)
            print('Source archives:', archives)
            raise RuntimeError('Không xác định được một bộ source mới. Mở scr-lab02 trong Input, kiểm tra src/optuna_dense.py và config/optuna_dense.json; đặt SOURCE_ROOT hoặc SOURCE_ZIP nếu có nhiều bản.')
    if SOURCE_ROOT is not None:
        SOURCE_ROOT = Path(SOURCE_ROOT)
        assert all((SOURCE_ROOT/p).is_file() for p in required), 'Dataset thiếu source Optuna mới; upload ZIP mới.'
        WORK.mkdir(parents=True, exist_ok=True)
        for name in ('src','config','tests'):
            shutil.copytree(SOURCE_ROOT/name, WORK/name, dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(SOURCE_ZIP) as archive:
            names = archive.namelist()
            assert all(p in names for p in required), 'ZIP thiếu source/config/test mới.'
            for member in archive.infolist():
                assert (WORK/member.filename).resolve().is_relative_to(WORK.resolve()), 'Đường dẫn ZIP không hợp lệ'
            WORK.mkdir(parents=True, exist_ok=True)
            archive.extractall(WORK)
assert all((WORK/p).is_file() for p in required)
sys.path.insert(0, str(WORK))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna>=4,<5', 'tensorboard', 'scikit-learn', 'pandas', 'matplotlib', 'pytest'], check=True)
print('Source ready:', WORK)


In [ ]:
import platform, importlib.metadata as metadata
import numpy as np
import pandas as pd
from IPython.display import display, Image, FileLink, Markdown
from src.delivery import write_json, file_hash
from src.experiment import seed_all
seed_all(42)
SETUP = WORK/'outputs/setup'
SETUP.mkdir(parents=True, exist_ok=True)
environment = {'python':platform.python_version(),'device':torch.cuda.get_device_name(0),
    'libraries':{name:metadata.version(name) for name in ('torch','torchvision','numpy','pandas','optuna','tensorboard')}}
write_json(SETUP/'environment.json', environment)
display(pd.Series(environment['libraries'], name='version').to_frame())


## 2. CIFAR10 và cấu hình chạy

Dữ liệu gốc là ảnh RGB 32×32, 10 lớp. Nạp dữ liệu official và kiểm tra checksum trước khi phân tích; dùng split stratified seed 42. Cấu hình head mới: 10 epoch, LR 0.001; fine-tune: 5 trial mỗi study, tối đa 20 epoch/trial.

In [ ]:
raw = list(Path('/kaggle/input').rglob('cifar-10-batches-py'))
if len(raw) == 1:
    DATA = raw[0].parent
else:
    DATA = WORK/'data'
    from torchvision.datasets import CIFAR10
    CIFAR10(str(DATA), train=True, download=True)
    CIFAR10(str(DATA), train=False, download=True)
os.environ['LAB02_DATA'] = str(DATA)
cfg = json.loads((WORK/'config/optuna_dense.json').read_text())
cfg['head_checkpoint'] = None
CONFIG = WORK/'config/kaggle_optuna.json'
CONFIG.write_text(json.dumps(cfg, indent=2))
from src.eda import read_raw
pool, labels, test_images, test_labels = read_raw(DATA)
assert len(labels) == 50000 and len(test_labels) == 10000
print('Data:', DATA)
print('Budget:', 2*cfg['trials_per_study'], 'trials, max', cfg['epochs'], 'epochs/trial')
del pool, test_images


## 3. EDA, split và đặc trưng mô tả

EDA kiểm tra nhãn/pixel, trùng ảnh chính xác, cân bằng lớp, ảnh trung bình và phân bố màu/không gian. Các feature thủ công ở Lab02 dùng để mô tả dữ liệu, không nối vào input DenseNet. EDA preview có cấu hình riêng; phần tiếp theo kiểm tra **transform 224×224 thực sự dùng bởi model**, tránh nhầm với preview resize 64 của EDA.

In [ ]:
from src import eda
EDA = eda.build(WORK, data_dir=DATA)
for name in ('structural_overview.csv','integrity_report.csv','class_distribution.csv','pixel_statistics.csv'):
    print(name)
    display(pd.read_csv(EDA/name))
duplicates = pd.read_csv(EDA/'duplicate_images.csv')
print('Exact duplicate records:', len(duplicates))
with np.load(EDA/'split_indices.npz', allow_pickle=False) as split:
    assert len(np.intersect1d(split['train'], split['val'])) == 0
    display(pd.DataFrame({'split':split.files,'samples':[len(split[k]) for k in split.files]}))


In [ ]:
for name in ('sample_images.png','class_distribution.png','class_mean_images.png','pixel_statistics.png','spatial_statistics.png','class_similarity.png','handcrafted_distributions.png','feature_correlation.png'):
    print(name)
    display(Image(filename=str(EDA/name)))


### Cách đọc EDA

Ảnh trung bình và similarity mô tả lớp, không phải attention hoặc confusion của model. Correlation feature không chứng minh feature importance. Trùng byte bằng 0 không loại trừ near-duplicate. Mean/std mô tả CIFAR10 không thay normalization cố định của pretrained ImageNet.

## 4. Kiểm chứng preprocessing thực tế

Transform lấy từ weights ImageNet: resize/crop theo contract rồi scale và normalize. Không augmentation. Train/validation/test cùng preprocessing; chỉ train loader shuffle. Lưu ảnh input trước/sau và tensor contract vào artifact.

In [ ]:
from src.data import image_transform
from torchvision.datasets import CIFAR10
import matplotlib.pyplot as plt
raw_dataset = CIFAR10(str(DATA), train=True, download=False)
raw_image, label = raw_dataset[0]
transform = image_transform('densenet121', cfg['image_size'])
tensor = transform(raw_image)
assert tensor.shape == (3,224,224) and torch.isfinite(tensor).all()
from src.models import SPECS
contract = SPECS['densenet121'][1].transforms()
recovered = (tensor*torch.tensor(contract.std)[:,None,None] + torch.tensor(contract.mean)[:,None,None]).clamp(0,1)
fig, axes = plt.subplots(1,2,figsize=(7,3))
axes[0].imshow(raw_image); axes[0].set_title('Raw CIFAR10 RGB 32x32')
axes[1].imshow(recovered.permute(1,2,0)); axes[1].set_title('Actual model input, de-normalized')
for ax in axes: ax.axis('off')
fig.tight_layout(); fig.savefig(SETUP/'actual_model_preprocessing.png',dpi=140); display(fig); plt.close(fig)
write_json(SETUP/'preprocessing.json',{'transform':str(transform),'shape':list(tensor.shape),
    'dtype':str(tensor.dtype),'normalization_mean':contract.mean,'normalization_std':contract.std,'augmentation':False})
print(transform)


## 5. DenseNet121 và phạm vi fine-tune

Khởi tạo từ ImageNet pretrained và thay classifier 10 lớp. Head-only chỉ train classifier. Fine-tune mở toàn bộ 16 dense layer block4, affine parameters norm5 và classifier. Các phần trước block4 đóng băng. BatchNorm chạy eval để giữ running mean/variance; affine parameters trong phần mở vẫn có gradient.

In [ ]:
from src.models import build_model, configure_trainable, parameter_counts, training_mode
probe = build_model('densenet121', pretrained=False)
configure_trainable(probe,'densenet121','head')
head_counts = parameter_counts(probe)
configure_trainable(probe,'densenet121','finetune','last_block')
training_mode(probe)
assert len(probe.features.denseblock4) == 16
assert all(not module.training for module in probe.modules() if isinstance(module,torch.nn.BatchNorm2d))
rows=[]
for name,p in probe.named_parameters():
    expected=name.startswith(('features.denseblock4.','features.norm5.','classifier.'))
    assert p.requires_grad == expected
    rows.append({'name':name,'parameters':p.numel(),'trainable':p.requires_grad})
parameters=pd.DataFrame(rows); parameters.to_csv(SETUP/'parameter_scope.csv',index=False)
(SETUP/'architecture.txt').write_text(str(probe))
display(pd.DataFrame([{'stage':'head',**head_counts},{'stage':'finetune',**parameter_counts(probe)}]))
display(parameters[parameters.trainable])
print('Model preview uses random initialization only to inspect architecture; training backend loads ImageNet weights.')
del probe


## 6. Optuna, LR chung/riêng và tiêu chí lựa chọn

Hai study độc lập, mỗi study 5 trial; TPE có 2 startup trials, không pruning. Shared tìm một LR; separate tìm LR backbone và LR head độc lập. LR log [1e-6,1e-3], weight decay log [1e-7,1e-3]. AdamW và cosine T_max=20, step cuối epoch. Không mặc định LR head gấp 10 backbone.

Mọi trial reset cùng checkpoint head vừa train. Validation loss làm objective. Checkpoint và early stopping dùng cùng min_delta=0.001, patience=4, không dừng trước epoch 6. Accuracy/F1 vẫn báo cáo. Ngân sách nhỏ chỉ cho kết luận khảo sát; hai best recipe không chứng minh riêng tác động nhân quả của LR grouping.

In [ ]:
display(pd.Series(cfg, name='configuration').to_frame())
print('Fine-tuning budget:',2*cfg['trials_per_study'],'trials')
print('Head training is additional:',cfg.get('head_epochs',10),'epochs')


## 7. Kiểm thử nhỏ trước training

Kiểm tra nhóm LR, phạm vi trainable và loss checkpoint/early stopping trên dữ liệu tổng hợp. Không tải weights hoặc chạy benchmark trong test.

In [ ]:
# Kiểm thử nhỏ; không tải pretrained weights hoặc train benchmark.
subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_optuna_dense.py', '-q'], cwd=WORK, check=True)


## 8. Train head mới và hai study

Cell này thực thi pipeline tổng hợp. Tiến trình từng epoch/trial hiện bên dưới và lưu vào run, gồm TensorBoard, SQLite, history và checkpoints. Không có trial nào đọc test để tối ưu. Cả hai winner được khóa trước đánh giá test; cuối cùng load hai model độc lập và predict lại toàn bộ test.

Nếu training lỗi, cell vẫn cố đóng gói partial outputs để bảo toàn bằng chứng. Run chưa hoàn tất không được coi là kết quả cuối.

In [ ]:
# TRAINING: chạy một lần; finally vẫn đóng gói partial artifacts nếu training lỗi.
from src.optuna_dense import run
try:
    RUN = run(CONFIG, work=WORK, device_name='cuda')
    print('Completed:', RUN)
finally:
    if (WORK/'outputs').exists():
        package = shutil.make_archive('/kaggle/working/lab02_optuna_outputs', 'zip', WORK, 'outputs')
        print('Artifact ZIP:', package)


## 9. Kết quả head và tiến trình search

Các cell sau chỉ đọc artifact. Chọn checkpoint head mới bằng validation loss trước khi so sánh hai study. Trạng thái COMPLETE/FAIL và số epoch thực cần được đọc cùng ngân sách tối đa.

In [ ]:
registry = json.loads((WORK/'outputs/selected_optuna.json').read_text())
RUN = WORK/registry['run']
assert json.loads((RUN/'status.json').read_text())['status']=='completed'
print('Selected run:', RUN.name)
head_history=pd.read_csv(RUN/'head_selected/history.csv')
display(head_history)
display(pd.Series(json.loads((RUN/'head_selected/training_summary.json').read_text())).to_frame('head summary'))
winners=json.loads((RUN/'best_parameters.json').read_text())
for mode in ('shared','separate'):
    print(mode)
    display(pd.read_csv(RUN/mode/'trials.csv'))
    display(pd.Series(winners[mode]).to_frame('selected trial'))


## 10. So sánh LR chung và LR riêng

Đọc loss, accuracy và macro metrics cùng validation objective. Đường cong train/validation giúp nhận diện overfit; kết quả test cao không phủ nhận khoảng cách train–validation. Không chọn lại cấu hình bằng kết quả test.

In [ ]:
comparison=pd.read_csv(RUN/'comparison.csv')
display(comparison)
display(Image(filename=str(RUN/'learning_curves.png')))
from src.delivery import write_json
observations=[]
for mode in ('shared','separate'):
    history=pd.read_csv(RUN/mode/f"trial_{winners[mode]['trial']:02d}"/'history.csv')
    note={'mode':mode,'epochs':len(history),'train_accuracy_last':float(history.train_accuracy.iloc[-1]),
        'val_accuracy_last':float(history.val_accuracy.iloc[-1]),'min_val_loss':float(history.val_loss.min()),
        'last_val_loss':float(history.val_loss.iloc[-1]),'objective_checkpoint_loss':winners[mode]['validation_loss']}
    observations.append(note)
display(pd.DataFrame(observations))
write_json(RUN/'comparison_observations.json',observations)


## 11. Confusion matrix, ROC/PR OvR và mẫu dự đoán

Mỗi phương án có một hình ROC/PR gồm 10 lớp theo One-vs-Rest. Confusion matrix hiển thị số đếm, màu chuẩn hóa theo lớp thật. Lưới chọn tối đa 8 đúng + 8 sai phục vụ phân tích; tỷ lệ trong lưới không phải accuracy. Đọc tên lớp theo thứ tự CIFAR10 và metric tương ứng; AP không đồng nhất với trapezoidal PR-AUC.

In [ ]:
from src.eda import CLASSES
print(dict(enumerate(CLASSES)))
for mode in ('shared','separate'):
    print(mode)
    evaluation=RUN/mode/'evaluation'
    display(pd.read_csv(evaluation/'classification_report.csv'))
    display(pd.read_csv(evaluation/'roc_pr_metrics.csv'))
    for name in ('confusion_matrix.png','roc_pr.png','prediction_examples.png'):
        display(Image(filename=str(evaluation/name)))
    display(pd.read_csv(evaluation/'prediction_examples.csv'))


## 12. Save/load và kiểm chứng toàn bộ test

Backend đã tạo model độc lập, load checkpoint, predict lại đủ test và kiểm tra probability tolerance atol=1e-6, rtol=1e-5. Bảng/hình dưới đây thuộc model sau load, không phải chỉ đọc ảnh từ trước load.

In [ ]:
for mode in ('shared','separate'):
    evaluation=RUN/mode/'evaluation'
    verify=json.loads((evaluation/'roundtrip.json').read_text())
    assert verify['passed'] and verify['samples']==10000
    print(mode,verify)
    display(pd.Series(json.loads((evaluation/'loaded_test_metrics.json').read_text())).to_frame('loaded model test'))
    display(Image(filename=str(evaluation/'loaded/prediction_examples.png')))
    display(pd.read_csv(evaluation/'loaded/prediction_examples.csv'))


## 13. ResNet18 — giữ thiết kế thực nghiệm cũ

Bài Lab02 có cả hai model. Chỉ DenseNet chuyển sang Optuna shared/separate LR. ResNet tái sử dụng config/completed.json, lọc models=['resnet18']: 6 head trial (3 LR × 2 batch), 10 epoch/head trial; 9 fine candidate cũ, tối đa 30 epoch, early stopping cũ và chọn bằng validation accuracy/loss/time. Không thêm augmentation/dropout. Chi phí này bổ sung ngoài thời gian DenseNet. Cell training dưới đây chỉ chạy ResNet, không gọi lại DenseNet.

In [ ]:
from src.experiment import run_suite
RESNET_REGISTRY = WORK/'outputs/selected_resnet.json'
if RESNET_REGISTRY.exists():
    RESNET_RUN = WORK/json.loads(RESNET_REGISTRY.read_text())['run']
    assert json.loads((RESNET_RUN/'status.json').read_text())['status'] == 'completed'
    print('Reuse completed ResNet run:',RESNET_RUN)
else:
    resnet_cfg=json.loads((WORK/'config/completed.json').read_text())
    resnet_cfg['models']=['resnet18']
    resnet_cfg['profile']='resnet18_original_recipe'
    RESNET_CONFIG=WORK/'config/kaggle_resnet_original.json'
    write_json(RESNET_CONFIG,resnet_cfg)
    try:
        RESNET_RUN=run_suite(RESNET_CONFIG,work_dir=WORK,device_override='cuda')
        write_json(RESNET_REGISTRY,{'run':str(RESNET_RUN.relative_to(WORK))})
    finally:
        package=shutil.make_archive('/kaggle/working/lab02_optuna_outputs','zip',WORK,'outputs')
        print('Partial/full package:',package)


In [ ]:
resnet_result=pd.read_csv(RESNET_RUN/'comparison.csv')
display(resnet_result)
for stage in ('head_selected','finetune'):
    evaluation=RESNET_RUN/'resnet18'/stage
    print('ResNet18',stage)
    display(pd.read_csv(evaluation/'history.csv'))
    for name in ('confusion_matrix.png','prediction_examples.png'):
        display(Image(filename=str(evaluation/name)))
display(Image(filename=str(RESNET_RUN/'resnet18/learning_curves.png')))
# Recipe comparison: selected checkpoints use different validation objectives.
combined=pd.concat([resnet_result.assign(recipe='ResNet original'),comparison.assign(model='densenet121',stage='finetune',recipe='DenseNet Optuna')],ignore_index=True)
combined.to_csv(WORK/'outputs/current_model_comparison.csv',index=False)
display(combined[['model','stage','recipe','accuracy','loss','macro_f1']])
print('ResNet selection uses accuracy/loss bands; DenseNet uses loss. This is a recipe comparison, not an isolated architecture ablation.')


## 13. Kết luận và bàn giao artifacts

Viết kết luận dựa trên bảng đã chạy: phương án nào có validation loss thấp hơn, test metrics chênh bao nhiêu, có dấu hiệu overfit hay không. Không kết luận chắc chắn từ 5 trial và một seed. Thí nghiệm chưa có augmentation/dropout. Cần xét thêm calibration/regularization nếu loss tăng trong khi accuracy ổn định.

Artifact ZIP chứa checkpoint, SQLite và console logs: giữ local, không push toàn bộ lên Git. Tải cả notebook đã chạy bằng giao diện Kaggle sau Save Version. Đóng gói lại ở cell cuối để bao gồm những bảng nhận xét được tạo sau training.

In [ ]:
# Update manifest after analysis cells added artifacts, then package the entire evidence.
write_json(RUN/'manifest.json',{str(p.relative_to(RUN)):file_hash(p) for p in RUN.rglob('*') if p.is_file() and p.name!='manifest.json'})
package=shutil.make_archive('/kaggle/working/lab02_optuna_outputs','zip',WORK,'outputs')
print('Artifact ZIP:',package)
display(FileLink(package))
